# 00 - Import CSV with All Data
**The CSV is expected to be in this format**
- ID and absolute paths to niftis are critical
```
+-----+----------------------------+--------------+--------------+--------------+
| ID  | Nifti_File_Path            | Covariate_1  | Covariate_2  | Covariate_3  |
+-----+----------------------------+--------------+--------------+--------------+
| 1   | /path/to/file1.nii.gz      | 0.5          | 1.2          | 3.4          |
| 2   | /path/to/file2.nii.gz      | 0.7          | 1.4          | 3.1          |
| 3   | /path/to/file3.nii.gz      | 0.6          | 1.5          | 3.5          |
| 4   | /path/to/file4.nii.gz      | 0.9          | 1.1          | 3.2          |
| ... | ...                        | ...          | ...          | ...          |
+-----+----------------------------+--------------+--------------+--------------+
```

Import Data

In [ ]:
# Specify the path to your CSV file containing NIFTI paths
input_csv_path = '/Volumes/OneTouch/01p_Schmahmann_SCA_Atrophy/results/optimzation/optimized_master_list_fibfiltered.csv'
sheet = None

In [ ]:
from calvin_utils.permutation_analysis_utils.statsmodels_palm import CalvinStatsmodelsPalm
# Instantiate the PalmPrepararation class
cal_palm = CalvinStatsmodelsPalm(input_csv_path=input_csv_path, output_dir=None, sheet=sheet)
# Call the process_nifti_paths method
data_df = cal_palm.read_and_display_data()
data_df


# 01 - Preprocess Your Data

**Handle NANs**
- Set drop_nans=True is you would like to remove NaNs from data
- Provide a column name or a list of column names to remove NaNs from

In [ ]:
data_df.columns

In [ ]:
drop_list = ['selected_Nifti_File_Path']

In [ ]:
data_df = cal_palm.drop_nans_from_columns(columns_to_drop_from=drop_list)
data_df

**Drop Row Based on Value of Column**

Define the column, condition, and value for dropping rows
- column = 'your_column_name'
- condition = 'above'  # Options: 'equal', 'above', 'below'

In [ ]:
data_df.columns

Set the parameters for dropping rows

In [ ]:
column = 'selected'  # The column you'd like to evaluate
condition = 'not'  # The condition to check ('equal', 'above', 'below', 'not')
value = 1 # The value to drop if found

In [ ]:
data_df, other_df = cal_palm.drop_rows_based_on_value(column, condition, value)
display(data_df)

# 02 - Run fiber filtering
- nifti_col - this is the column in the dataframe you provided with all the seed files in it
- patient_id_col - column in dataframe with patient ids in them
- fiber_atlas_path - the path to the atlas generated in notebook 00, or a .tt.ck from DSI Studio
- reference_nifti_path - this is the mask, typically the mask representing the space your seed niftis are in (i.e. MNI152.nii.gz)
- mode - determines how fibers are filtered. binary sets fibers to 1 if anything hits them. Max sets them to max value of what hit them. mean is the average. Read docs for more. 
- binarize - binarizes the input images
- threshold - numeric value provided if binarize=True

In [ ]:
data_df.columns

In [ ]:
from calvin_utils.neuroimaging_utils.tract_utils.filter_fibers import filter_fibers

runner = filter_fibers(
    df=data_df,
    nifti_col="Nifti_File_Path",
    patient_id_col="subid",  # optional, remove if not available
    fiber_atlas_path="/Volumes/OneTouch/01p_Schmahmann_SCA_Atrophy/connectomic/um1_fibers.npz",
    reference_nifti_path="/Users/cu135/Software_Local/calvin_utils_project/circuit_pyper/resources/mni_icbm152_t1_tal_nlin_sym_09c_mask.nii",
    out_dir="/Volumes/OneTouch/01p_Schmahmann_SCA_Atrophy/Schmahmann_BIDS",
    mode="max",
    binarize=False,
    threshold=None
)

updated_df = runner.run()

Save the CSV with fiber paths added to it

In [ ]:
from pathlib import Path
p = Path(input_csv_path)
out_path = p.with_name(f"{p.stem}_um1fibfiltered.csv")
updated_df.to_csv(out_path, index=False)